## ML_1035: Flash Attention (Tiled)

**Difficulty**: Hard | **TorchCode**: #25

### Problem
Implement tiled Flash Attention with online softmax. Process Q in blocks; for each Q-block iterate over K/V blocks. Track running max and sum to rescale the accumulator without materializing the full attention matrix.

### Formula
For each new block with max $m_{\text{new}}$:
$$\text{acc} \leftarrow \text{acc} \cdot e^{m_{\text{old}} - m_{\text{new}}} + e^{s - m_{\text{new}}} V, \quad \text{out} = \text{acc} / \text{row\_sum}$$

In [ ]:
import torch
import math

def flash_attention(Q, K, V, block_size=32):
    B, S, D = Q.shape
    output = torch.zeros_like(Q)
    for i in range(0, S, block_size):
        qi = Q[:, i:i+block_size]
        bs_q = qi.shape[1]
        row_max = torch.full((B, bs_q, 1), float('-inf'), device=Q.device)
        row_sum = torch.zeros(B, bs_q, 1, device=Q.device)
        acc = torch.zeros(B, bs_q, D, device=Q.device)
        for j in range(0, S, block_size):
            kj = K[:, j:j+block_size]
            vj = V[:, j:j+block_size]
            scores = torch.bmm(qi, kj.transpose(1, 2)) / math.sqrt(D)
            block_max = scores.max(dim=-1, keepdim=True).values
            new_max = torch.maximum(row_max, block_max)
            correction = torch.exp(row_max - new_max)
            exp_scores = torch.exp(scores - new_max)
            acc = acc * correction + torch.bmm(exp_scores, vj)
            row_sum = row_sum * correction + exp_scores.sum(dim=-1, keepdim=True)
            row_max = new_max
        output[:, i:i+block_size] = acc / row_sum
    return output

In [ ]:
import torch
import math

# --- Test 1: Matches standard attention ---
torch.manual_seed(0)
B, S, D = 2, 16, 8
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(D)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 2: Non-aligned block size ---
torch.manual_seed(42)
B, S, D = 1, 7, 4
Q, K, V = torch.randn(B,S,D), torch.randn(B,S,D), torch.randn(B,S,D)
out = flash_attention(Q, K, V, block_size=3)
scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(D)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 3: Block size invariant ---
torch.manual_seed(0)
Q, K, V = torch.randn(1,12,8), torch.randn(1,12,8), torch.randn(1,12,8)
assert torch.allclose(flash_attention(Q, K, V, block_size=4),
                      flash_attention(Q, K, V, block_size=6), atol=1e-4)

# --- Test 4: Gradient flow ---
Q = torch.randn(1, 8, 4, requires_grad=True)
K = torch.randn(1, 8, 4, requires_grad=True)
V = torch.randn(1, 8, 4, requires_grad=True)
flash_attention(Q, K, V, block_size=4).sum().backward()
assert Q.grad is not None

print('All tests passed!')